# 01 — RealCause TarNet Training

Trains a TarNet generative model on the Sepsis dataset using **identical seed
and hyperparameters** to the original `02_cdv_modeling.ipynb`.

Saves the trained checkpoint to `artifacts/realcause_model/medium_seed_420/model.pt`.

**Run once.** The experiment notebook (02) loads this checkpoint and never retrains.

In [ ]:
import sys, os
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import numpy as np
import pandas as pd
import torch
import json
import warnings
warnings.filterwarnings('ignore')

from data.sepsis import load_sepsis
from cdv_utils.generator_validation import (
    train_multiple_models, analyze_model_performance, select_best_model
)

ARTIFACTS_DIR = 'cdv_experiments/sepsis/artifacts'
DATASET_PATH  = os.path.join(ARTIFACTS_DIR, 'sepsis_cases.csv')
SAVEROOT      = os.path.join(ARTIFACTS_DIR, 'realcause_model')

assert os.path.exists(DATASET_PATH), (
    f'sepsis_cases.csv not found at {DATASET_PATH}. '
    'Run 00_data_preparation.ipynb first.'
)

os.makedirs(SAVEROOT, exist_ok=True)
print(f'Dataset: {DATASET_PATH}')
print(f'Model will be saved to: {SAVEROOT}')

Dataset: revised_experiment/sepsis/artifacts\sepsis_cases.csv
Model will be saved to: revised_experiment/sepsis/artifacts\realcause_model


## 1. Load Data

In [2]:
# Identical to original NB02: load from artifacts/
sepsis_data = load_sepsis(data_format='numpy', dataroot=ARTIFACTS_DIR)
w, t, y = sepsis_data

original_df = pd.read_csv(DATASET_PATH)
w_cols = [c for c in original_df.columns if c not in ['t', 'y', 'y0', 'y1', 'ite', 'variant']]

print(f'Covariates (w): {w.shape}')
print(f'Treatment (t): {t.shape}, mean={np.mean(t):.3f}')
print(f'Outcome (y): {y.shape}')

Loading sepsis dataset from revised_experiment/sepsis/artifacts\sepsis_cases.csv
Covariates (w): (810, 30)
Treatment (t): (810,), mean=0.089
Outcome (y): (810,)


## 2. Train TarNet (same seed=420, same architecture as original NB02)

In [3]:
# Identical hyperparameters to original NB02 Section 3.1
INITIAL_SEED   = 420
training_seeds = [INITIAL_SEED]

print(f'Training RealCause model (seed={INITIAL_SEED})...')
print('Architecture: medium (3 layers, 128 hidden units, ReLU)')
print('Splits: train=50%, val=10%, test=40%')

results, best_model_selector_dict = train_multiple_models(
    w, t, y,
    w_cols=w_cols,
    saveroot=SAVEROOT,
    seeds=training_seeds
)

print('\nTraining complete!')

Training RealCause model (seed=420)...
Architecture: medium (3 layers, 128 hidden units, ReLU)
Splits: train=50%, val=10%, test=40%

Training medium architecture:
- Hidden layers: 3
- Hidden units: 128
- Activation: ReLU
n_train: 405	n_val: 81	n_test: 324
test_idxs:  (324,)


  0%|          | 1/400 [00:00<01:05,  6.06it/s]

Iteration 10 valid loss 2.418264389038086
saving best-val-loss model
Iteration 20 valid loss 2.4024510383605957
saving best-val-loss model
Iteration 30 valid loss 2.385331630706787
saving best-val-loss model


  1%|          | 3/400 [00:00<00:41,  9.45it/s]

Iteration 40 valid loss 2.3649020195007324
saving best-val-loss model
Iteration 50 valid loss 2.337759494781494
saving best-val-loss model


...

Iteration 1470 valid loss -0.32322537899017334
Iteration 1480 valid loss -0.30409279465675354
Iteration 1490 valid loss -0.3068442642688751
Iteration 1500: 0.07223530858755112 -1.2168684005737305
Iteration 1500 valid loss -0.28760242462158203


 29%|██▉       | 117/400 [01:00<01:56,  2.43it/s]

Iteration 1510 valid loss -0.2953440248966217
Iteration 1520 valid loss -0.2759971618652344
Iteration 1530 valid loss -0.3239732086658478
early stopping criterion reached. Ending experiment.


 29%|██▉       | 117/400 [01:00<02:26,  1.93it/s]


loading best-val-loss model (early stopping checkpoint)

Training complete!


## 3. Performance Metrics

In [4]:
performance_df = analyze_model_performance(results, SAVEROOT)
print('Model performance metrics:')
display(performance_df)

best_model = select_best_model(results, best_model_selector_dict, criterion='medium')
print(f'\nSelected model: {best_model.__class__.__name__}')
print(f'Checkpoint: {SAVEROOT}/medium_seed_420/model.pt')

Model performance metrics:


,Architecture,Seed,Univariate: Y KS p-value,Univariate: T KS p-value,Univariate: Y ES p-value,Univariate: T ES p-value,Univariate: Y Wasserstein,Univariate: T Wasserstein,Multivariate: Wasserstein1 p-value,Multivariate: Wasserstein2 p-value,Multivariate: kNN p-value,Multivariate: Energy p-value,Multivariate: Friedman-Rafsky p-value,ATE,Model Path
0,medium,seed_420,0.28183,0.952218,0.241516,0.879783,816.970534,0.015432,0.02,0.152,0.117,0.012,0.235,-2324.199307,revised_experiment/sepsis/artifacts\realcause_...



Selected model: TarNet
Checkpoint: revised_experiment/sepsis/artifacts\realcause_model/medium_seed_420/model.pt


## 4. Quick Sanity Check — ATE and Distributions

In [5]:
ate = best_model.ate().item()
noisy_ate = best_model.noisy_ate(seed=INITIAL_SEED).item()
print(f'Model ATE (deterministic): {ate:.4f}')
print(f'Model noisy ATE (sampled): {noisy_ate:.4f}')

# Verify checkpoint exists
ckpt = os.path.join(SAVEROOT, 'medium_seed_420', 'model.pt')
assert os.path.exists(ckpt), f'Checkpoint not found: {ckpt}'
print(f'\nCheckpoint confirmed: {ckpt}')
print('Ready for experiment notebook (02_experiment.ipynb).')

Model ATE (deterministic): -2309.4808
Model noisy ATE (sampled): -2314.6443

Checkpoint confirmed: revised_experiment/sepsis/artifacts\realcause_model\medium_seed_420\model.pt
Ready for experiment notebook (02_experiment.ipynb).
